In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gp
from shapely.geometry import point
import matplotlib.pyplot as plt

In [2]:
DATA_DIR = Path.cwd().parent / "data"
df = pd.read_csv(DATA_DIR / "raw" / "NYC.csv")
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
df.head(5)

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.98,40.77,-73.96,40.77,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.98,40.74,-74.00,40.73,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.98,40.76,-74.01,40.71,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.01,40.72,-74.01,40.71,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.97,40.79,-73.97,40.78,N,435


In [4]:
df.isna().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [5]:
df.isnull().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [6]:
df['trip_duration'] = df['trip_duration']/60

In [7]:
df['trip_duration'].describe()

count   1458644.00
mean         15.99
std          87.29
min           0.02
25%           6.62
50%          11.03
75%          17.92
max       58771.37
Name: trip_duration, dtype: float64

The minimum trip duration is 0.02 minutes which is an outlier , and shouldn't exist,
The maximum trip duration is 58771.37 , which means that the trip  took 40 days to finish , which cannot be possible in Newyork city, it is an outlier and should be 
removed

In [8]:
df = df[df['trip_duration']>=1]
df = df[df['trip_duration'] <= 120] 

After checking the data for any Null values , there is no missing data in the dataset

In [9]:
df.describe()

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
count,1447796.00,1447796.00,1447796.00,1447796.00,1447796.00,1447796.00,1447796.00
mean,1.54,1.67,-73.97,40.75,-73.97,40.75,14.01
std,0.50,1.31,0.07,0.03,0.07,0.04,10.89
min,1.00,0.00,-121.93,34.36,-121.93,32.18,1.00
25%,1.00,1.00,-73.99,40.74,-73.99,40.74,6.68
50%,2.00,1.00,-73.98,40.75,-73.98,40.75,11.08
75%,2.00,2.00,-73.97,40.77,-73.96,40.77,17.93
max,2.00,9.00,-61.34,51.88,-61.34,43.92,119.85


In [10]:
print(f"Longitude {df['pickup_longitude'].min()}, Latitude {df['pickup_latitude'].min()}")

Longitude -121.93334197998048, Latitude 34.35969543457031


In [11]:
print(f"Longitude {df['dropoff_longitude'].min()}, Latitude {df['dropoff_latitude'].min()}")

Longitude -121.9333038330078, Latitude 32.1811408996582


The data referenced showed that the longitude and latitude point to california , which is an outlier in the NYC dataset
Now making sure that the  longitude and latitude data are in the range of NYC longitude and latitude

In [12]:
mask = (
    (df['dropoff_longitude'].between(-74.26, -73.70)) &
    (df['dropoff_latitude'].between(40.49, 40.92)) & 
    (df['pickup_longitude'].between(-74.26,-73.70)) & 
    (df['pickup_latitude'].between(40.49,40.92))
)
df_clean = df[mask]
print(len(df), len(df_clean)) 

1447796 1446659


After Removing some outlier's the cleaned data length is 145327
Now saving the final cleaned data

In [ ]:
boroughs = gp.read_file("/Users/aangphurbasherpa/Desktop/NYC Taxi Duration/data/processed/boundary.geojson")
gdf = gp.GeoDataFrame(df, geometry=gp.points_from_xy(df['dropoff_longitude'], df['dropoff_latitude']), crs="EPSG:4326")
gdf = gdf.rename_geometry('points_geom')
boroughs['geometry'] = boroughs['geometry'].buffer(0)
boroughs = boroughs.to_crs("EPSG:4326")

result = gp.sjoin(gdf, boroughs[['BoroName', 'geometry']], how='left', predicate='within')
print(result['BoroName'].value_counts(dropna=False))

BoroName
Manhattan        1281213
Brooklyn           77003
Queens             74146
Bronx               9271
Staten Island        369
Name: count, dtype: int64


In [16]:
result = result.drop(columns=['geometry'], errors='ignore')
result.to_csv(DATA_DIR / "processed" / "cleaned_taxi_data.csv", index=False)